# Alpamayo-v1.5 Phase 3c — Cascade FM-loss top-N → full-trajectory ADE/FDE

Cross-cutting consumer: globs every `results/bfa_*.parquet` written by `bfa_demo.ipynb` (Phase 1), `bfa_vlm_grad.ipynb` (Phase 3a), and `bfa_vlm_random.ipynb` (Phase 3b), picks the top-N flips by `post_loss` from each, replays them through `sample_trajectories_from_data_with_vlm_rollout`, and computes `minADE / meanADE / minFDE / meanFDE` against the ground-truth waypoints.

**Phase 2a (KV cache) results are skipped.** The cache is rebuilt from scratch on every inference call inside `sample_trajectories...`, so a flip on the saved prefill cache wouldn't reach the rollout. A cache-aware cascade would need to patch the inference loop, which is intentionally out of scope.

Cost: ~5–10 s per top-N rollout × ~50 per source × ~3 sources ≈ **20–30 min**.
Output: `results/cascade_ade_<source_stem>.parquet` per input results file, plus a comparison boxplot.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to("cuda")
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
SCENE_IDX = 774
clip_id = clip_ids[SCENE_IDX]
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    "cuda",
)

gt_xy_traj = data["ego_future_xyz"].numpy()[..., :2]
if gt_xy_traj.ndim == 3:
    gt_xy_traj = gt_xy_traj[0]  # squeeze leading batch dim if present
print("gt_xy_traj shape:", gt_xy_traj.shape)

In [ ]:
# Smoke (i): clean rollout reproduces a baseline ADE — establishes the
# reference number that all flipped rollouts will be compared against.
ROLLOUT_KWARGS = dict(num_traj_samples=16, top_p=0.95, temperature=0.5,
                     max_generation_length=128)

roll_clean = bfa.run_rollout(model, model_inputs, seed=42, **ROLLOUT_KWARGS)
pred_clean = roll_clean["pred_xyz"].numpy()
if pred_clean.ndim == 4:
    pred_clean = pred_clean[0]
baseline_metrics = bfa.compute_traj_metrics(pred_clean[..., :2], gt_xy_traj)
print("baseline rollout metrics:", baseline_metrics)

In [ ]:
# Discover every BFA results parquet. Filter to the scenes we care about.
results_dir = Path("results")
all_parquets = sorted(results_dir.glob("bfa_*.parquet"))
candidates = [p for p in all_parquets if f"scene{SCENE_IDX}" in p.name and "vlm_kv" not in p.name and "kv" not in p.name]
print("will cascade these sources:")
for p in candidates:
    print(" ", p.name)

In [ ]:
TOP_N = 50
all_cascade_dfs = []
for src in candidates:
    df = pd.read_parquet(src)
    if "module" not in df.columns:
        print(f"skip {src.name} — no 'module' column (likely a cache-target dump)")
        continue

    # Build target_lookup once per source (only the modules referenced).
    unique_names = df["module"].unique().tolist()
    target_lookup = {}
    for name in unique_names:
        try:
            mod = model.get_submodule(name)
        except AttributeError:
            continue
        if isinstance(mod, torch.nn.Linear):
            target_lookup[name] = mod
    print(f"{src.name}: {len(target_lookup)}/{len(unique_names)} module names resolved")

    cascade = bfa.cascade_top_n_to_rollout(
        df, model, model_inputs, gt_xy_traj,
        target_lookup=target_lookup,
        top_n=TOP_N,
        seed=42,
        rollout_kwargs=ROLLOUT_KWARGS,
    )
    cascade["source"] = src.stem
    out_path = results_dir / f"cascade_ade_{src.stem}.parquet"
    cascade.to_parquet(out_path)
    print(f"  → {out_path}: {len(cascade)} rows cascaded")
    all_cascade_dfs.append(cascade)

In [ ]:
# Comparison: ADE distributions per source phase. The clean baseline minADE/meanADE
# is overlaid as a horizontal reference line for context.
if not all_cascade_dfs:
    print("no cascade results — run bfa_demo / bfa_vlm_grad / bfa_vlm_random first")
else:
    combined = pd.concat(all_cascade_dfs, ignore_index=True)
    combined_finite = combined[combined["n_finite"] > 0]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["minADE_m", "meanADE_m"]):
        combined_finite.boxplot(column=metric, by="source", ax=ax, showfliers=False)
        ax.axhline(baseline_metrics[metric], color="r", linestyle="--",
                   label=f"clean {metric} = {baseline_metrics[metric]:.3f} m")
        ax.set_title(metric)
        ax.set_xlabel("")
        ax.set_ylabel("meters")
        ax.legend(loc="upper left", fontsize=8)
        ax.tick_params(axis="x", rotation=20)
    plt.suptitle(f"Cascaded top-{TOP_N} flips → ADE per source phase (scene {SCENE_IDX})")
    plt.tight_layout()
    plt.savefig(results_dir / f"cascade_ade_compare_scene{SCENE_IDX}.png", dpi=120)
    plt.show()

    print("\nPer-source ADE summary:")
    print(combined_finite.groupby("source")[["minADE_m", "meanADE_m", "minFDE_m", "meanFDE_m"]].describe())